# QM 640 Capstone — Colab runner
**Correcting for Circle-Rate Anchoring in Indian Residential Property Valuation** — Hemant Kumar Sahay (proprietary; see LICENSE)

Use Colab when the data are too large for a laptop. Data live in **Google Drive** (`QM640_DATA_ROOT`), code comes from the **private** GitHub repo. Run one RQ per cell and review before moving on.

## 1. Mount Google Drive (data storage)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.environ['QM640_DATA_ROOT'] = '/content/drive/MyDrive/qm640_data'   # raw/interim/processed live here
os.makedirs(os.environ['QM640_DATA_ROOT'], exist_ok=True)

## 2. Clone the private repository
Store a fine-grained GitHub token (read-only, this repo only) in Colab **Secrets** as `GH_TOKEN`, and the hashing salt as `QM640_HASH_SALT`.

In [ ]:
from google.colab import userdata
token = userdata.get('GH_TOKEN')
os.environ['QM640_HASH_SALT'] = userdata.get('QM640_HASH_SALT')
!git clone https://{token}@github.com/hemantsahay/qm640-registry-valuation-india.git
%cd qm640-registry-valuation-india
!pip -q install -r requirements.txt osmnx geopy deep-translator

## 3. (Optional) Refresh OpenStreetMap stations and locality coordinates

In [ ]:
!python -m src.features.osm_features

## 4. ETL — parse Marathi Index-II, translate, mask (DPDP), clean, build features
Upload `index2_raw.csv` from your laptop collection to `qm640_data/raw/` first. For a dry run add `--synthetic` after generating validation data with `!python -m src.ingest.synthetic_validation`.

In [ ]:
!python -m src.pipeline --stage etl

## 5. Exploratory data analysis

In [ ]:
!python -m src.pipeline --stage eda
from IPython.display import Image, display
import glob
for f in sorted(glob.glob('outputs/eda/fig_*.png')): display(Image(f))

## 6. RQ1 — bunching at the ASR threshold  ⏸ review before continuing

In [ ]:
!python -m src.pipeline --stage rq1
for f in sorted(glob.glob('outputs/rq1/fig_*.png')): display(Image(f))
print(open('outputs/rq1/summary.json').read())

## 7. RQ2 — hedonic OLS  ⏸

In [ ]:
!python -m src.pipeline --stage rq2

## 8. RQ3 — XGBoost vs OLS on held-out micro-markets  ⏸

In [ ]:
!python -m src.pipeline --stage rq3

## 9. RQ4 — anchoring classifier and correction factor  ⏸

In [ ]:
!python -m src.pipeline --stage rq4

## 10. Save results back to GitHub (outputs + results log only — never raw data)

In [ ]:
!git add outputs docs/RESULTS_LOG.md && git commit -m 'Colab run' && git push